# 02 - Data Cleaning

Fixes from notebook 01: duplicates, price units, yield units, rainfall and temperature gaps, soil test gaps.
Output: `dataset/cleaned_dataset.csv`.

In [1]:
import pandas as pd
import numpy as np

raw = pd.read_csv('../dataset/raw_dataset.csv')
df = raw.drop_duplicates().copy()
print('rows before:', len(raw), ' after:', len(df))

rows before: 348456  after: 347760


## Price units

In [2]:
# kg prices are roughly 1/100 of quintal prices, so anything below 10% of the crop median is a per-kg entry
crop_median = df.groupby('crop').mandi_price.transform('median')
in_kg = df.mandi_price < 0.1 * crop_median
df['price_q'] = np.where(in_kg, df.mandi_price * 100, df.mandi_price)
print('converted rows:', in_kg.sum(), '(%.1f%%)' % (100 * in_kg.mean()))
df.groupby('crop').price_q.agg(['min', 'median', 'max']).round(0)

converted rows: 43512 (12.5%)
            min   median      max
crop                             
cotton   6571.0  11567.0  17854.0
gram     4809.0   8602.0  17065.0
soybean  4218.0   7077.0  11229.0
tur      5485.0  10581.0  18822.0
wheat    2032.0   4061.0   7230.0


## Yield units

In [3]:
key = ['taluka', 'crop', 'season_year']
# same value on all 46 weekly rows of a season, so decide at season level
assert df.groupby(key).yield_reported.nunique().max() == 1

ratio = df.yield_reported / df.groupby('crop').yield_reported.transform('median')
df['yield_q_ha'] = df.yield_reported
df.loc[ratio > 30, 'yield_q_ha'] = df.yield_reported / 100    # entered as kg/ha
df.loc[ratio < 0.3, 'yield_q_ha'] = df.yield_reported * 10    # entered as t/ha

season = df.drop_duplicates(key)
r = ratio[season.index]
print('seasons fixed:', ((r > 30) | (r < 0.3)).sum(), 'of', len(season))
after = df.yield_q_ha / df.groupby('crop').yield_q_ha.transform('median')
print('yield / median after fix:', round(after.min(), 2), 'to', round(after.max(), 2))

seasons fixed: 281 of 7560
yield / median after fix: 0.39 to 1.66


## Rainfall and temperature

Rain is recorded per taluka per week, and the same weekly reading appears under every crop. Missing values are filled with:
1. the average of the other readings from the same district in the same week (known at that time, so no future information), then
2. the district's average for that calendar week, using only years up to 2021 (the training years).

I also keep a flag column so the model knows which values were filled.

In [4]:
df['rain_was_missing'] = df.rain_mm.isna().astype(int)

same_week = df.groupby(['district', 'abs_week']).rain_mm.transform('mean')
train_part = df[df.season_year <= 2021]
clim = train_part.groupby(['district', 'calendar_week']).rain_mm.mean().rename('rain_clim')
df = df.join(clim, on=['district', 'calendar_week'])

before = df.rain_mm.isna().sum()
df['rain_mm'] = df.rain_mm.fillna(same_week).fillna(df.rain_clim)
print('rain missing before:', before, ' after:', df.rain_mm.isna().sum())

# same idea for temperature
t_week = df.groupby(['district', 'abs_week']).tmax_c.transform('mean')
t_clim = train_part.groupby(['district', 'calendar_week']).tmax_c.mean().rename('t_clim')
df = df.join(t_clim, on=['district', 'calendar_week'])
df['tmax_c'] = df.tmax_c.fillna(t_week).fillna(df.t_clim)
df = df.drop(columns=['rain_clim', 't_clim'])

rain missing before: 20312  after: 0


In [5]:
# quick check of the filling method: hide some known monsoon values (more of the heavy ones, like the real gaps),
# fill them again, and see how far off we are
rng = np.random.default_rng(0)
stations = df.drop_duplicates(['taluka', 'abs_week'])
stations = stations[stations.rain_was_missing == 0]
st = stations[stations.calendar_week.between(23, 40)]
hide = (rng.random(len(st)) < (0.05 + 0.3 * np.clip(st.rain_mm / 60, 0, 1))).values
truth = st.rain_mm[hide]

others = stations.groupby(['district', 'abs_week']).rain_mm.agg(['sum', 'count'])
j = st[hide].join(others, on=['district', 'abs_week'])
neighbour = (j['sum'] - j.rain_mm) / (j['count'] - 1)
clim_only = st[hide].join(train_part.groupby(['district', 'calendar_week']).rain_mm.mean().rename('c'),
                          on=['district', 'calendar_week']).c
print('MAE neighbour fill  : %.2f mm/week, bias %.2f' % ((neighbour - truth).abs().mean(), (neighbour - truth).mean()))
print('MAE climatology fill: %.2f mm/week, bias %.2f' % ((clim_only - truth).abs().mean(), (clim_only - truth).mean()))

MAE neighbour fill  : 50.94 mm/week, bias -26.46
MAE climatology fill: 48.52 mm/week, bias -26.36


Honest result: neither fill is good. Both are off by about 50 mm in a missing week and both under-estimate, because the missing weeks are the heavy ones. The neighbour average is not better than the climatology here since rain differs a lot between talukas within a week.

I keep the neighbour-then-climatology order (it costs nothing), but I do not trust single weeks. The model gets the `rain_was_missing` flag and the share of missing weeks so far, and it mostly uses 4-week, 8-week and season-to-date totals, where the weekly errors partly average out. This is a known weakness and it goes in the model card.

## Soil test

Soil is tested once a year per taluka. Gaps are filled with the taluka's value from the previous year; if there is none, the district median of the training years. A flag column is kept.

In [6]:
soil_cols = ['soil_N', 'soil_P', 'soil_K', 'soil_pH', 'soil_OC']
df['soil_was_missing'] = df.soil_N.isna().astype(int)

ty = df.drop_duplicates(['taluka', 'season_year'])[['taluka', 'district', 'season_year'] + soil_cols]
ty = ty.sort_values(['taluka', 'season_year'])
ty[soil_cols] = ty.groupby('taluka')[soil_cols].ffill()
med = ty[ty.season_year <= 2021].groupby('district')[soil_cols].median()
for c in soil_cols:
    ty[c] = ty[c].fillna(ty.district.map(med[c]))

df = df.drop(columns=soil_cols).merge(ty[['taluka', 'season_year'] + soil_cols], on=['taluka', 'season_year'], how='left')
print('soil still missing:', df[soil_cols].isna().sum().sum())

soil still missing: 0


In [7]:
df = df.drop(columns=['mandi_price', 'yield_reported'])
df = df.sort_values(['taluka', 'crop', 'season_year', 'season_week']).reset_index(drop=True)
assert (df.groupby(['taluka', 'crop', 'season_year']).size() == 46).all()
df.to_csv('../dataset/cleaned_dataset.csv', index=False)
print(df.shape)
df.isna().sum()[df.isna().sum() > 0]

(347760, 23)
Series([], )


Cleaned file saved. `season_mandi_arrivals_qpha` and the arrivals columns are still in the file on purpose; I check them for leakage in notebook 04 and drop them there.